# kafka_spark.API.ipynb
## API Reference: Apache Kafka and Spark Structured Streaming
**Author**: Aashish Vinod  
**Course**: DATA605 Spring 2026  

## Why I built this

As a grad student I have used Kafka and Spark in coursework before, but always with pre-configured environments. For this project I wanted to wire everything together from scratch inside Docker — producer, broker, consumer, and Spark all talking to each other.

The trickiest part was Docker networking. When Kafka runs inside a container, you cannot reach it at `localhost:9092` from another container. You have to use the service name defined in `docker-compose.yml`, which is `kafka:29092`. This caused a `NoBrokersAvailable` error that took me a while to track down.

This notebook is my API reference — I document each component individually so anyone reading it can understand what each piece does before seeing the full pipeline in `kafka_spark.example.ipynb`.

## Notebook structure
1. Kafka Producer API
2. Kafka Consumer API
3. Utility Functions API
4. Spark Structured Streaming API
5. Windowed Aggregations API

## 1. Imports

In [1]:
# Standard imports - I'm using kafka-python for the Kafka client
# and findspark to initialize PySpark from inside Jupyter
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka_spark_utils import (
    generate_stock_event, generate_stock_stream,
    serialize_event, deserialize_event,
    compute_moving_average, check_price_alert,
    format_kafka_summary, STOCK_SYMBOLS, BASE_PRICES,
)
print('All imports successful!')
print(f'Stocks: {STOCK_SYMBOLS}')
print(f'Base prices: {BASE_PRICES}')


All imports successful!
Stocks: ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']
Base prices: {'AAPL': 175.0, 'GOOGL': 140.0, 'MSFT': 380.0, 'AMZN': 185.0, 'TSLA': 250.0}


## 2. Kafka Producer API
KafkaProducer sends messages to a Kafka topic.

In [2]:
# Note: I'm connecting to kafka:29092 NOT localhost:9092
# This is because both Jupyter and Kafka run inside Docker containers
# and Docker uses the service name 'kafka' as the hostname internally.
# This was the main networking issue I had to debug.
producer = KafkaProducer(
    bootstrap_servers='kafka:29092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
)
print(f'Producer connected to Kafka successfully')
print(f'Bootstrap servers: kafka:29092')


Producer created successfully
Bootstrap servers: localhost:9092


In [3]:
# Send a single event
event = generate_stock_event('AAPL')
print(f'Event to send:')
print(json.dumps(event, indent=2))
future = producer.send('stock-prices', key='AAPL', value=event)
producer.flush()
print('Message sent successfully!')


Event to send:
{
  "symbol": "AAPL",
  "price": 178.09,
  "volume": 349,
  "timestamp": "2026-05-05T19:05:14.493077",
  "change_pct": 1.7675
}
Message sent successfully!


## 3. Kafka Consumer API
KafkaConsumer reads messages from a Kafka topic.

In [4]:
# Initialize KafkaConsumer
consumer = KafkaConsumer(
    'stock-prices',
    bootstrap_servers='kafka:29092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='api-demo-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=3000,
)
messages = []
for msg in consumer:
    messages.append(msg.value)
consumer.close()
print(f'Total messages received: {len(messages)}')
if messages:
    print(f'Sample message: {json.dumps(messages[0], indent=2)}')


Total messages received: 1
Sample message: {
  "symbol": "AAPL",
  "price": 178.09,
  "volume": 349,
  "timestamp": "2026-05-05T19:05:14.493077",
  "change_pct": 1.7675
}


## 4. Utility Functions API

In [5]:
# generate_stock_event(symbol)
event = generate_stock_event('MSFT')
print('generate_stock_event("MSFT"):')
print(json.dumps(event, indent=2))


generate_stock_event("MSFT"):
{
  "symbol": "MSFT",
  "price": 380.8,
  "volume": 7075,
  "timestamp": "2026-05-05T19:05:20.753507",
  "change_pct": 0.2096
}


In [6]:
# compute_moving_average(prices, window)
prices = [100, 102, 101, 103, 105, 104, 106, 108, 107, 109]
ma5 = compute_moving_average(prices, window=5)
print(f'Prices:          {prices}')
print(f'MA5 (window=5):  {ma5}')


Prices:          [100, 102, 101, 103, 105, 104, 106, 108, 107, 109]
MA5 (window=5):  [None, None, None, None, 102.2, 103.0, 103.8, 105.2, 106.0, 106.8]


In [7]:
# check_price_alert(price, symbol, threshold_pct)
alert1 = check_price_alert(180.0, 'AAPL', threshold_pct=1.5)
alert2 = check_price_alert(175.5, 'AAPL', threshold_pct=1.5)
print('Alert for AAPL at $180.0 (base=$175.0):')
print(json.dumps(alert1, indent=2) if alert1 else 'No alert triggered')
print('\nAlert for AAPL at $175.5 (base=$175.0):')
print(json.dumps(alert2, indent=2) if alert2 else 'No alert triggered')


Alert for AAPL at $180.0 (base=$175.0):
{
  "symbol": "AAPL",
  "alert": "Price moved UP by 2.86%",
  "current_price": 180.0,
  "base_price": 175.0,
  "change_pct": 2.8571,
  "timestamp": "2026-05-05T19:05:20.777588"
}

Alert for AAPL at $175.5 (base=$175.0):
No alert triggered


In [8]:
# serialize_event / deserialize_event
event = generate_stock_event('TSLA')
serialized = serialize_event(event)
deserialized = deserialize_event(serialized)
print(f'Original:     {event}')
print(f'Serialized:   {serialized[:60]}...')
print(f'Deserialized: {deserialized}')
print(f'Match: {event == deserialized}')


Original:     {'symbol': 'TSLA', 'price': 248.75, 'volume': 6269, 'timestamp': '2026-05-05T19:05:20.791181', 'change_pct': -0.4987}
Serialized:   b'{"symbol": "TSLA", "price": 248.75, "volume": 6269, "timesta'...
Deserialized: {'symbol': 'TSLA', 'price': 248.75, 'volume': 6269, 'timestamp': '2026-05-05T19:05:20.791181', 'change_pct': -0.4987}
Match: True


## 5. Spark Structured Streaming API

In [9]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('StockMarketAPI') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-80dd7457-6044-4d08-9aaa-ab9d2e846912;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 551ms :: artifacts dl 22ms
	:: modules in us

Spark version: 3.5.1


In [10]:
# Define schema for stock events
stock_schema = StructType([
    StructField('symbol', StringType(), True),
    StructField('price', DoubleType(), True),
    StructField('volume', IntegerType(), True),
    StructField('timestamp', StringType(), True),
    StructField('change_pct', DoubleType(), True),
])
print('Stock event schema:')
print(stock_schema.simpleString())


Stock event schema:
struct<symbol:string,price:double,volume:int,timestamp:string,change_pct:double>


In [11]:
# Read from Kafka as streaming DataFrame
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:29092') \
    .option('subscribe', 'stock-prices') \
    .option('startingOffsets', 'earliest') \
    .load()
print('Kafka streaming DataFrame schema:')
kafka_df.printSchema()


Kafka streaming DataFrame schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [12]:
# Parse JSON values and apply windowed aggregation
parsed_df = kafka_df \
    .select(F.from_json(F.col('value').cast('string'), stock_schema).alias('data')) \
    .select('data.*') \
    .withColumn('event_time', F.to_timestamp('timestamp'))

windowed = parsed_df \
    .withWatermark('event_time', '10 seconds') \
    .groupBy(F.window('event_time', '30 seconds', '10 seconds'), 'symbol') \
    .agg(
        F.avg('price').alias('avg_price'),
        F.max('price').alias('max_price'),
        F.min('price').alias('min_price'),
        F.count('price').alias('event_count'),
    )
print('Windowed aggregation schema:')
windowed.printSchema()


Windowed aggregation schema:
root
 |-- window: struct (nullable = true)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- event_count: long (nullable = false)



In [13]:
spark.stop()
print('Spark session stopped.')


Spark session stopped.
